# Dt 谱 × meta-gradient 金标准
**回答的问题：Dt 作为"下游贡献度代理"有没有价值。**

meta-grad（一阶）：∂L_task/∂w(t) ∝ −⟨∇L_task, ∇L_diff(t)⟩，是该 t 对下游任务真实增益的直接度量。Dt 谱与 meta-grad 谱的 Spearman/Pearson 相关性越高，Dt 越可信。

支持对多个 checkpoint 批量测量并汇总相关性演化。

In [ ]:
import os, sys
import torch

# ---- 路径：notebook 在 visual/ 下，切到 mmsegmentation 作为工作目录 ----
MMSEG = os.path.abspath('../tasks/Segmentation/mmsegmentation')
os.chdir(MMSEG)
sys_path_added = []
for p in [MMSEG, os.path.join(MMSEG, 'eden')]:
    import sys
    if p not in sys.path:
        sys.path.insert(0, p)

# eden 包在服务器上部署名为 mydiffae：本地别名兼容
import eden, eden.diffae_mixin, eden.pcgrad_mixin
sys.modules.setdefault('mydiffae', eden)
sys.modules.setdefault('mydiffae.diffae_mixin', eden.diffae_mixin)
sys.modules.setdefault('mydiffae.pcgrad_mixin', eden.pcgrad_mixin)

# 注册 adapter（FCNWithDiffAEPlusPCGrad）；已注册过则跳过
try:
    import mmseg.models.segmentors.eden_adapter_fcn  # noqa
except KeyError:
    pass

sys.path.insert(0, os.path.abspath('../visual'))
import probe_core as pc


In [ ]:
# ---- 配置：改成你的 config 与 ckpt 列表 ----
CFG = 'configs/fcn/fcn_r50-d8_4xb4-80k_ade20k-512x512_diffae_tweight.py'
CKPTS = [
    'work_dirs/eden/fcn_diffae_tweight/iter_20000.pth',
    'work_dirs/eden/fcn_diffae_tweight/iter_40000.pth',
    'work_dirs/eden/fcn_diffae_tweight/iter_80000.pth',
]
OUT = 'work_dirs/offline_probe/dt_meta'
NUM_IMAGES = 16
os.makedirs(OUT, exist_ok=True)

In [ ]:
# 数据只取一次，所有 ckpt 共用（同图同分布，跨 ckpt 可比）
model, cfg = pc.build_model(CFG)   # 先建一次只为取数
batch = pc.build_val_batch(model, cfg, NUM_IMAGES)
del model; torch.cuda.empty_cache()

In [ ]:
rows = []
for ckpt in CKPTS:
    model, _ = pc.build_model(CFG, ckpt)
    tag = os.path.splitext(os.path.basename(ckpt))[0]
    dt = pc.measure_dt_spectrum(model, batch)
    mg = pc.measure_meta_grad_spectrum(model, batch)
    fig, (rho_s, p_s, rho_p) = pc.plot_dt_vs_meta(
        dt, mg, title=tag,
        save_path=os.path.join(OUT, f"{tag}_dt_vs_meta.png"))
    rows.append((tag, rho_s, p_s, rho_p))
    print(f"{tag}: Spearman ρ={rho_s:.3f} (p={p_s:.3f}), Pearson r={rho_p:.3f}")
    del model; torch.cuda.empty_cache()
fig  # 显示最后一张

In [ ]:
# 跨 ckpt 汇总：相关性随训练进度演化
import matplotlib.pyplot as plt
if len(rows) > 1:
    tags = [r[0] for r in rows]
    fig, ax = plt.subplots(figsize=(8, 4))
    ax.plot(tags, [r[1] for r in rows], 'o-', label='Spearman ρ')
    ax.plot(tags, [r[3] for r in rows], 's-', label='Pearson r')
    ax.axhline(0, color='k', ls='--', lw=0.8)
    ax.set_ylim(-1, 1); ax.grid(alpha=0.3); ax.legend()
    ax.set_title('Dt ~ meta-grad correlation across checkpoints')
    plt.setp(ax.get_xticklabels(), rotation=30, ha='right')
    fig.savefig(os.path.join(OUT, 'summary.png'), dpi=150, bbox_inches='tight')
    fig